# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()



,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))


!git clone https://github.com/jasontpv/DS2002FA26.git
cat = pd.read_csv('DS2002FA26/notebooks/06-midterm-walmart/data/product_catalog.csv')

x = cat['product_name'].str.lower().str.contains('pop', na=False)
pop = cat[x]
print(f"Rows that look like Pop-Tart products: {len(pop)}")
pop

Cloning into 'DS2002FA26'...
remote: Enumerating objects: 369, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 369 (delta 122), reused 99 (delta 71), pack-reused 192 (from 1)
Receiving objects: 100% (369/369), 3.76 MiB | 14.31 MiB/s, done.
Resolving deltas: 100% (208/208), done.
Rows that look like Pop-Tart products: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [28]:
is_poptart = cat['product_name'].str.lower().str.contains(r'pop.?tart', na=False)
is_straw = cat['product_name'].str.lower().str.contains('strawberry', na=False)

straw = cat[is_poptart & is_straw]
other_pop = cat[is_poptart & ~is_straw]

print("Strawberry Pop-Tart rows:", len(straw))
display(straw[['sku', 'product_name']])

print("Pop-Tart rows that are a different product:", len(other_pop))


Strawberry Pop-Tart rows: 4


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


Pop-Tart rows that are a different product: 0


**Pop-Tart rows that are a different product:** _..._ None are present. All 4 Pop-Tart rows (`PT-12`, `SKU#459812`, `POPTART-STRAW`, `PT_STRAWBERRY`) are strawberry, as the empty `other_pop` output above shows. If any other flavor appeared, it would stay separate from the strawberry group and not be merged into it but for now there are zero products as not strawberry.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [25]:
CANON = 'PT-12'
sku_map = {sku: CANON for sku in ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'PT_STRAWBERRY']}

cat['canonical_sku'] = cat['sku'].replace(sku_map)

print(cat.loc[cat['sku'] != cat['canonical_sku'], ['sku', 'canonical_sku']])

print(cat['canonical_sku'].isna().sum(), "NaNs")
print(cat['sku'].nunique(), "->", cat['canonical_sku'].nunique(), "unique SKUs")

             sku canonical_sku
1     SKU#459812         PT-12
2  POPTART-STRAW         PT-12
3  PT_STRAWBERRY         PT-12
0 NaNs
45 -> 42 unique SKUs


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [29]:
pt12 = cat['canonical_sku'] == 'PT-12'

print("Before:\n", cat.loc[pt12, 'category'].value_counts())

cat['category_clean'] = cat['category'].str.strip().str.title()
print("\nAfter strip + title-case:\n", cat.loc[pt12, 'category_clean'].value_counts())

cat.loc[pt12, 'category_clean'] = cat.loc[pt12, 'category_clean'].replace({'Breakfast Foods': 'Breakfast'})
print("\nAfter mapping:\n", cat.loc[pt12, 'category_clean'].value_counts())

Before:
 category
Breakfast          2
Breakfast Foods    1
breakfast          1
Name: count, dtype: int64

After strip + title-case:
 category_clean
Breakfast          3
Breakfast Foods    1
Name: count, dtype: int64

After mapping:
 category_clean
Breakfast    4
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [17]:
cat['normalized'] = (cat['product_name'].str.lower()
                     .str.replace(r'[^a-z ]', ' ', regex=True)
                     .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (cat.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

,sku
normalized,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
aid first kit,"[FAID-01, FIRSTAID-KIT]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [31]:
orig_rows = len(pd.read_csv('DS2002FA26/notebooks/06-midterm-walmart/data/product_catalog.csv'))

assert cat['canonical_sku'].notna().all(), "canonical_sku has nulls"
assert len(cat) == orig_rows, f"row count changed: {orig_rows} -> {len(cat)}"

print("Checks passed:", len(cat), "rows, 0 nulls in canonical_sku")

Checks passed: 45 rows, 0 nulls in canonical_sku


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Monopoly: BG-MONO, MONOPOLY-01, and GAME-MONO (“Monopoly Board Game” and “Board Game - Monopoly”) are one product under three SKUs. Sorting the words in the name makes them share a key, and I would map all three to one canonical SKU, e.g. BG-MONO.

Light beer: BEER-24 and LTBEER24 have the identical name, “Light Beer (24-pack)”. The category labels differ (“Alcoholic Beverages” vs “ALCOHOL”), which needs its own normalization step. I would map both to BEER-24.

b)

Paper towels. PTOWEL-6 is a 4 pack and PAPERTOWEL6 is a 12-pack, though the names are identical and both SKUs say “6.” I can’t tell from the data whether this is one product with sloppy pack-size data or two different sellable items. If I merge them, unit costs, sales and reorder quantities blend two pack sizes. I would ask the category manager/merchandising team whether the two pack sizes are sold as separate items, and supply chain/inventory whether they are stocked and replenished separately.